# Day 2 — Chat Completions API across providers

Calling the Chat Completions API directly via `requests`, then via the `openai` Python client, then using the same client to talk to Gemini and Ollama via OpenAI-compatible endpoints.

## Load environment variables and validate the OpenAI key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found — check your .env file.")
elif not api_key.startswith("sk-proj-"):
    print("API key found, but doesn't start with 'sk-proj-' — verify it's the correct key.")
else:
    print("API key looks good.")

## Calling the Chat Completions endpoint directly with `requests`

This is what the OpenAI Python client wraps under the hood — a plain HTTP POST.

In [ ]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}
    ]
}

payload

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()["choices"][0]["message"]["content"]

## Same call using the `openai` Python client

Cleaner syntax, same underlying HTTP call.

In [ ]:
from openai import OpenAI

openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=[{"role": "user", "content": "Tell me a fun fact"}]
)

response.choices[0].message.content

## Talking to Gemini via the OpenAI-compatible endpoint

Many providers expose an OpenAI-compatible Chat Completions endpoint, so the same client library works by overriding `base_url` and `api_key`.

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No Google API key found — skip the next cell or add GOOGLE_API_KEY to your .env")
elif not google_api_key.startswith("AIz"):
    print("Google API key found but doesn't start with 'AIz' — verify it.")
else:
    print("Google API key looks good.")

In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[{"role": "user", "content": "Tell me a fun fact"}]
)

response.choices[0].message.content

## Calling a local model via Ollama

Ollama runs locally and also exposes an OpenAI-compatible endpoint at `http://localhost:11434/v1`.

If the next cell doesn't return `Ollama is running`, start the server in a terminal: `ollama serve`.

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
# Pull a small Llama model. Use llama3.2:1b on smaller machines.
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
response = ollama.chat.completions.create(
    model="llama3.2",
    messages=[{"role": "user", "content": "Tell me a fun fact"}]
)

response.choices[0].message.content

In [ ]:
# Try DeepSeek's distilled Qwen model
!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(
    model="deepseek-r1:1.5b",
    messages=[{"role": "user", "content": "Tell me a fun fact"}]
)

response.choices[0].message.content